# 05 · Out-of-sample forecasts: running the models week by week

**The question in one sentence.** If we had been standing at the end of each Friday from Dec 2012 to Oct 2026 and fitted every model only on the data we had *then*, what would each model have forecast for the next week?

**How we answer it.** An *expanding window*: at each Friday (the **origin**) we re-fit M1, M2, M3 and the benchmarks on all data up to that Friday, forecast the next Friday (the **target**), then move one week forward. That gives **723 forecasts per model** (313 validation + 365 test + 45 holdout weeks). Notebook 06 scores them.

**Why there are four stages.** The full run is long, and some settings (how many restarts, which M3 variant) must be chosen *before* we look at the test years. So we work in stages, and each stage only looks at the validation years (targets 2013–2018):

| stage | what it does | how long |
|---|---|---|
| `timing` | fits the first 10 origins to measure how slow each model is | a few minutes |
| `validation` | runs all 313 validation origins, then we review the fits | minutes (the timing stage gives the estimate) |
| `freeze` | writes down the agreed settings (`spec_freeze.json`) | seconds |
| `full` | runs all 723 origins (resumes from validation if nothing changed), assembles the files and checks them | the timing stage gives the estimate (test runs suggest well under an hour) |
| `all` | re-displays the saved review tables and checks; starts no long run (only refits the two GARCH-X models for one check) | a minute |

**Choosing the stage.** Set the environment variable `BRENT_STAGE` (default `validation`), or edit the default in the `STAGE = ...` line of the setup cell. A cell of another stage prints "skipped". `BRENT_STAGE=all` starts no long run: it only re-displays the review tables and checks from files that are already saved (so it is useful after the full run; before that, cells whose files do not exist yet just print "skipped"). It still refits the two small GARCH-X models for the likelihood-ordering check (a few seconds).

**The final committed version** of this notebook is produced with `BRENT_STAGE=full` after the full run has finished: the run cell then sees `forecasts_main.csv` and skips the long job, and all checks, including the resume test, run.

**Changing a setting is safe.** Every saved week carries a fingerprint of the settings (the *settings hash*). If you change a setting in `config.py` (or the code changes how models are estimated), the old checkpoint no longer matches and is discarded automatically: the next run starts again from the first origin. You never need to delete files by hand.

**Rules that never change:** nothing from 2019 on is used to choose any setting; the settings are frozen before the `full` stage; every config change is made **by hand** in `config.py` (this notebook only tells you which line to set).

## 0 · Setup

Loads the packages, picks the stage, loads the weekly panel and builds the three Markov-switching models from `config.py`. `FORCE_OOS = False` means: never throw away saved forecasts unless you change it on purpose.

In [ ]:
# Reload src/brent/*.py automatically when we edit them.
%load_ext autoreload
%autoreload 2

import json
import os
import subprocess
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import brent
from brent import config as C        # every constant and frozen decision
from brent import paths              # file locations + size budget
from brent import plots              # figure style + save_fig
from brent import ms                 # Markov-switching specs and fitting
from brent import oos                # the expanding-window loop, checkpoints, assembly
from brent import insample as I      # garchx_lr (used for a sanity check)
from brent import benchmarks as B    # benchmark forecasts (timed here)

np.random.seed(C.SEED)
plots.setup_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# ── FLAGS ─────────────────────────────────────────────────────────────────────────────────────
# In VS Code: change "timing" below to the stage you want to run next (timing -> validation -> freeze -> full).
STAGE = os.environ.get("BRENT_STAGE", "full")     # timing | validation | freeze | full | all (display only)
FORCE_OOS = False                                    # True = rebuild forecasts_main.csv even if it exists
assert STAGE in ("timing", "validation", "freeze", "full", "all"), f"unknown BRENT_STAGE {STAGE!r}"


def on(stage):
    """True if cells of `stage` should run: the stage matches, or STAGE == 'all' (review cells only)."""
    return STAGE == stage or STAGE == "all"


# ── DATA AND MODELS ───────────────────────────────────────────────────────────────────────────
panel = pd.read_csv(paths.PROCESSED / "weekly_panel.csv", parse_dates=["date"])
S = ms.make_specs(C.PHI_SWITCHING, variant=C.M3_VARIANT)     # M1, M2, M3 from config.py
MS = [S["M1"], S["M2"], S["M3"]]
VALID_ORIGINS = oos.origins(C.VALID[0], C.VALID[1])          # 313 origins: 2012-12-28 -> 2018-12-21

print(f"STAGE = {STAGE}   FORCE_OOS = {FORCE_OOS}   SPEC_FROZEN = {C.SPEC_FROZEN}")
print(f"panel: {len(panel):,} weeks, {panel['date'].min().date()} -> {panel['date'].max().date()}")
print(f"PHI_SWITCHING={C.PHI_SWITCHING}  M3_VARIANT={C.M3_VARIANT}  CONTROLS_ONLY_NH={C.CONTROLS_ONLY_NH}  MAX_TVTP_COEF={C.MAX_TVTP_COEF}")
print(f"N_RESTARTS={C.N_RESTARTS}  RESTART_EVERY={C.RESTART_EVERY}  SEARCH_REPS_FIRST={C.SEARCH_REPS_FIRST}")
print(f"benchmarks: {C.BENCHMARKS_MAIN}")
print(f"validation origins: {len(VALID_ORIGINS)} ({VALID_ORIGINS[0].date()} -> {VALID_ORIGINS[-1].date()})")

STAGE = freeze   FORCE_OOS = False   SPEC_FROZEN = True
panel: 1,140 weeks, 2005-01-07 -> 2026-11-06
PHI_SWITCHING=False  M3_VARIANT=constrained  CONTROLS_ONLY_NH=True  MAX_TVTP_COEF=50
N_RESTARTS=3  RESTART_EVERY=13  SEARCH_REPS_FIRST=30
benchmarks: ('GARCH_t', 'HAR', 'RW', 'HAR_h', 'HARX_C', 'HARX_CI', 'GARCHX_C', 'GARCHX_CI')
validation origins: 313 (2012-12-28 -> 2018-12-21)


In [2]:
# ── GUARD: φ MUST MATCH NOTEBOOK 03 ───────────────────────────────────────────────────────────
# Notebook 03 decided PHI_SWITCHING and saved it in rq1_decision.json. If config.py disagrees we stop here.
dec = json.loads((paths.FROZEN / "rq1_decision.json").read_text())
assert C.PHI_SWITCHING is not None, "PHI_SWITCHING is not set in config.py"
assert C.PHI_SWITCHING == dec["phi_switching"], \
    f"config.PHI_SWITCHING={C.PHI_SWITCHING} but rq1_decision.json says {dec['phi_switching']}"
print("OK: PHI_SWITCHING in config.py matches rq1_decision.json:", C.PHI_SWITCHING)

OK: PHI_SWITCHING in config.py matches rq1_decision.json: False


### Expected number of forecasts per model

Not every model starts in Dec 2012. The hourly benchmarks need hourly data, which exists only from June 2013. The hourly HAR models then wait for one year of it (`HAR_MIN_ROWS` = 52 weekly rows, so they start around Sep 2014), because a regression fitted on a handful of weeks is meaningless. GARCH-X waits for its 2-year scaling window (June 2015). So the number of forecast weeks differs by model. The helper below computes the expected number from `oos.first_origin`, and the checks at the end of the notebook compare against it.

In [3]:
# ── EXPECTED TARGETS PER MODEL ────────────────────────────────────────────────────────────────
ALL_ORIGINS = oos.origins(C.VALID[0], C.FREEZE_DATE)          # all 723 origins (target Friday minus 7 days)


def expected_targets(model):
    """Number of forecast weeks we expect for `model`: all origins on or after its first origin."""
    fo = oos.first_origin(model) if model not in ("M1", "M2", "M3") else None
    return sum(1 for o in ALL_ORIGINS if fo is None or o >= fo)


EXPECTED = {m: expected_targets(m) for m in ["M1", "M2", "M3", *C.BENCHMARKS_MAIN]}
print(f"total origins: {len(ALL_ORIGINS)} ({ALL_ORIGINS[0].date()} -> {ALL_ORIGINS[-1].date()})")
for m, n in EXPECTED.items():
    fo = oos.first_origin(m) if m in C.BENCHMARKS_MAIN else None
    print(f"  {m:9s} {n:4d} forecast weeks" + (f"   (first origin {fo.date()})" if fo is not None and n != len(ALL_ORIGINS) else ""))

total origins: 723 (2012-12-28 -> 2026-10-30)
  M1         723 forecast weeks
  M2         723 forecast weeks
  M3         723 forecast weeks
  GARCH_t    723 forecast weeks
  HAR        723 forecast weeks
  RW         723 forecast weeks
  HAR_h      636 forecast weeks   (first origin 2014-08-29)
  HARX_C     636 forecast weeks   (first origin 2014-08-29)
  HARX_CI    636 forecast weeks   (first origin 2014-08-29)
  GARCHX_C   596 forecast weeks   (first origin 2015-06-05)
  GARCHX_CI  596 forecast weeks   (first origin 2015-06-05)


## 1 · Stage `timing`: how slow is each model?

**Why.** Each Markov-switching fit takes seconds, and we need 723 of them per model. Before choosing how many random restarts to use, we measure the time.

**What the next cell does.**
1. Runs the **first 10 validation origins** (Dec 28 2012 to Mar 1 2013) with all models. It never touches a test-window origin and is never resumed (`force=True`).
2. The first fit is **cold** (no starting values, a long search); the other nine are **warm** (start from last week's parameters), which is how the real run works.

In [4]:
# ── RUN THE TIMING ORIGINS ────────────────────────────────────────────────────────────────────
# oos.origins("2013-01-04", "2013-03-08") = the 10 origins 2012-12-28 ... 2013-03-01 (targets 2013-01-04 ... 2013-03-08).
# force=True: always start from scratch, so the timing is never mixed with an older run.
if STAGE == "timing":
    TIMING_ORIGINS = oos.origins("2013-01-04", "2013-03-08")
    t0 = time.perf_counter()
    oos.run_oos(panel, "timing", MS, origins=TIMING_ORIGINS, force=True)
    print(f"timing run: {len(TIMING_ORIGINS)} origins in {time.perf_counter() - t0:.0f} s")
else:
    print("skipped: this cell belongs to stage timing (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage timing (STAGE is 'freeze')


**What the next cell does.** Prints seconds per fit for each model: the cold first fit separately from the warm fits (origins 1 to 9).

In [5]:
# ── SECONDS PER FIT: COLD VS WARM ─────────────────────────────────────────────────────────────
if on("timing"):
    if not (paths.CHECKPOINTS / "timing_fitlog.csv").exists():
        print("skipped: " + str(paths.CHECKPOINTS / "timing_fitlog.csv") + " does not exist yet (run its stage first)")
    else:
        tlog = pd.read_csv(paths.CHECKPOINTS / "timing_fitlog.csv", dtype={"settings_hash": str})
        tlog["origin"] = pd.to_datetime(tlog["origin"])
        first = tlog["origin"].min()
        ran = tlog.dropna(subset=["seconds"])                          # hourly benchmarks have no row yet (skipped)
        cold = ran[ran["origin"] == first].set_index("model")["seconds"]
        warm = ran[ran["origin"] > first].groupby("model")["seconds"].agg(["mean", "max"])
        timing_tab = pd.DataFrame({"cold_s (origin 0)": cold, "warm_mean_s": warm["mean"], "warm_max_s": warm["max"]}).round(3)
        display(timing_tab)
        skipped = tlog[tlog["restart_reason"].astype(str).str.startswith("skipped")]["model"].unique()
        print("not timed here (they start later because hourly data starts later):", list(skipped))

        # restart weeks seen in the timing run (anything other than 'scheduled' = a sanity flag triggered a restart)
        msl = tlog[tlog["model"].isin(["M1", "M2", "M3"])]
        print("restart reasons in the timing run:", msl["restart_reason"].fillna("").replace("", "(no restart)").value_counts().to_dict())
else:
    print("skipped: this cell belongs to stage timing (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage timing (STAGE is 'freeze')


**What the next cell does.** Times the **hourly benchmarks** (HAR_h, HARX_C, HARX_CI, GARCHX_C, GARCHX_CI). They cannot run at the first 10 origins (no hourly data yet), so we time them at their own first origin (cold) and the next week (warm).

In [6]:
# ── TIME THE HOURLY BENCHMARKS AT THEIR OWN FIRST ORIGIN ──────────────────────────────────────
# bm_state carries warm starts (like the real loop). First call = cold, second call (one week later) = warm.
if STAGE == "timing":
    bm_time = {}
    for name in ("HAR_h", "HARX_C", "HARX_CI", "GARCHX_C", "GARCHX_CI"):
        fo = oos.first_origin(name)
        state = {"iran_var": C.IRAN_VAR}
        c = B.model_forecast(name, panel, fo, state)["seconds"]
        w = B.model_forecast(name, panel, fo + oos.WEEK, state)["seconds"]
        bm_time[name] = {"first_origin": fo.date(), "cold_s": round(c, 3), "warm_s": round(w, 3)}
    bm_time = pd.DataFrame(bm_time).T
    display(bm_time)
    GARCHX_COLD_MAX = float(bm_time.loc[["GARCHX_C", "GARCHX_CI"], "cold_s"].max())
else:
    print("skipped: this cell belongs to stage timing (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage timing (STAGE is 'freeze')


**What the next cell does (restart cost).** A *restart* means: after the warm fit, also fit again from a few random starting points and keep the best. On origin 2013-03-01, for each Markov-switching model, we time `fit_ms` with 0, 1, 3 and 5 restarts, starting from last week's parameters. The slope is the **seconds per restart**.

In [7]:
# ── RESTART COST ──────────────────────────────────────────────────────────────────────────────
# Warm start = the parameters stored for the origin BEFORE 2013-03-01 (like the real loop does).
if STAGE == "timing":
    if not (paths.CHECKPOINTS / "timing_params.csv").exists():
        print("skipped: " + str(paths.CHECKPOINTS / "timing_params.csv") + " does not exist yet (run its stage first)")
    else:
        from brent import design as D
        T_BENCH = pd.Timestamp("2013-03-01")
        tparams = pd.read_csv(paths.CHECKPOINTS / "timing_params.csv", dtype={"origin": str})
        done = sorted(pd.Timestamp(o) for o in tparams["origin"].unique())
        prev = [o for o in done if o < T_BENCH] or done                       # fall back to the last origin if needed
        state, _ = oos._rebuild_state(panel, MS, tparams, [prev[-1]])
        rest_rows, rest_cost = [], {}
        for spec in MS:
            dsn = D.build_design(panel, spec, T_BENCH)
            ms.fit_ms(dsn, spec, state[spec.name], 0, 0, C.SEED)            # throwaway warm-up fit (first call is slower)
            xs = [0, 1, 3, 5]
            secs = []
            for k in xs:
                t0 = time.perf_counter()
                ms.fit_ms(dsn, spec, state[spec.name], 0, k, C.SEED + oos.origin_index(T_BENCH))
                secs.append(time.perf_counter() - t0)
            slope = float(np.polyfit(xs, secs, 1)[0])                        # seconds added per extra restart
            rest_cost[spec.name] = max(slope, 0.0)
            rest_rows.append({"model": spec.name, **{f"{k} restarts (s)": round(s, 2) for k, s in zip(xs, secs)},
                              "seconds per restart": round(slope, 2)})
        display(pd.DataFrame(rest_rows))
else:
    print("skipped: this cell belongs to stage timing (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage timing (STAGE is 'freeze')


**What the next cell does (runtime estimate).** For each candidate `(N_RESTARTS, RESTART_EVERY)` it estimates the total time of the full run:

`total = 723 x (sum of warm seconds, all models) + (723 / RESTART_EVERY + extra) x N_RESTARTS x (sum of seconds per restart)`

"extra" is the number of unscheduled restarts triggered by the sanity checks: 10% of origins as a guess, or the share actually seen in the timing run if there was any. The hourly benchmarks start later, so we count them only for the weeks they run.

In [8]:
# ── RUNTIME TABLE ─────────────────────────────────────────────────────────────────────────────
if STAGE == "timing":
    if not (paths.CHECKPOINTS / "timing_fitlog.csv").exists():
        print("skipped: " + str(paths.CHECKPOINTS / "timing_fitlog.csv") + " does not exist yet (run its stage first)")
    else:
        N = len(ALL_ORIGINS)                                                    # 723
        warm_ms = float(warm.loc[["M1", "M2", "M3"], "mean"].sum())
        warm_bm = sum(warm.loc[m, "mean"] * EXPECTED[m] for m in warm.index if m in C.BENCHMARKS_MAIN)
        warm_bm += sum(bm_time.loc[m, "warm_s"] * EXPECTED[m] for m in bm_time.index)
        warm_total = warm_ms * N + warm_bm
        flagged = msl[(msl["origin"] > first) & (msl["restart_reason"].fillna("") != "") & (msl["restart_reason"].fillna("") != "scheduled")]
        seen = flagged["origin"].nunique() / max(msl["origin"].nunique() - 1, 1)
        extra_share = seen if seen > 0 else 0.10
        print(f"unscheduled-restart share used: {extra_share:.0%} ({'seen in the timing run' if seen > 0 else 'default guess'})")
        print(f"warm fits of all models, 723 origins: {warm_total / 3600:.2f} h")
        per_restart = sum(rest_cost.values())
        rows = []
        for n_rest, every in [(1, 26), (2, 13), (3, 13), (3, 26), (5, 13)]:
            total = warm_total + (N / every + extra_share * N) * n_rest * per_restart
            rows.append({"N_RESTARTS": n_rest, "RESTART_EVERY": every, "estimated hours": round(total / 3600, 2)})
        runtime_tab = pd.DataFrame(rows)
        display(runtime_tab)
        print(f"GARCH-X cold fit, worst case: {GARCHX_COLD_MAX:.2f} s")
        if runtime_tab.loc[(runtime_tab.N_RESTARTS == 1) & (runtime_tab.RESTART_EVERY == 26), "estimated hours"].iloc[0] > 3 or GARCHX_COLD_MAX > 2:
            print("STOP: even (1, 26) is above 3 h, or GARCH-X cold fits take more than 2 s. Talk to the team before going on.")
        print(f"\nNOW IN config.py: N_RESTARTS = {C.N_RESTARTS}, RESTART_EVERY = {C.RESTART_EVERY}")
        print("TARGET: about 2 h. Pick a row, then set the two lines by hand in src/brent/config.py (no need to restart the kernel).")
else:
    print("skipped: this cell belongs to stage timing (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage timing (STAGE is 'freeze')


### What this means
*(The code outputs saved in this notebook are from the last stage that was run; cells of other stages show "skipped". The numbers below come from each stage's own run and its log files.)*

- **Speed:** after the first week, each weekly fit takes under 0.1 seconds per model (M1 0.09 s, M2 0.07 s, M3 0.09 s). Only the very first fit, from scratch, takes 0.5–3 seconds. The benchmarks take 0.01 seconds or less.
- **Restarts are cheap:** one extra careful re-fit costs 0.08–0.25 seconds per model.
- **Decision:** we kept `N_RESTARTS = 3` and `RESTART_EVERY = 13` (3 extra re-fits every 13 weeks). The whole run was expected to take minutes, far below the 2-hour target.

## 2 · Stage `validation`: run the 313 validation origins

**What the next cell does.** Runs `oos.run_oos(panel, "main", MS, origins=<313 validation origins>)`: origins Dec 28 2012 to Dec 21 2018 (targets Jan 4 2013 to Dec 28 2018), all three models and all benchmarks. It saves after every week, so if it is interrupted you just run the same cell again and it **resumes**. A progress line prints every 50 origins. It never looks at 2019 or later.

Before you run it: the restart settings in `config.py` should be the ones you picked in section 1.

In [9]:
# ── RUN THE VALIDATION ORIGINS ────────────────────────────────────────────────────────────────
# run_oos refuses to run if SPEC_FROZEN is False and an origin's target is after 2018-12-28, so this cannot
# touch the test years. If settings in config.py changed since the last run, the saved checkpoint is ignored
# (the settings hash differs) and the run starts from the first origin.
if STAGE == "validation":
    t0 = time.perf_counter()
    oos.run_oos(panel, "main", MS, origins=VALID_ORIGINS)
    print(f"validation run finished in {(time.perf_counter() - t0) / 60:.1f} min")
else:
    print("skipped: this cell belongs to stage validation (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage validation (STAGE is 'freeze')


### Review (validation weeks only)

**What the next cells do.** Read `results/checkpoints/main_fitlog.csv` and print, for each Markov-switching model: how often the fit converged, how often it had to start over, how often the labels (calm/turbulent) were swapped back, how many fits were degenerate; the restarts; and the spread of the quantities behind the three safety thresholds (`SIGMA2_RATIO_MIN`, `MIN_DURATION`, `VAR_FLOOR_FRAC`).

In [10]:
# ── LOAD THE VALIDATION FITLOG ────────────────────────────────────────────────────────────────
if on("validation"):
    if not (paths.CHECKPOINTS / "main_fitlog.csv").exists():
        print("skipped: " + str(paths.CHECKPOINTS / "main_fitlog.csv") + " does not exist yet (run its stage first)")
    else:
        vlog = pd.read_csv(paths.CHECKPOINTS / "main_fitlog.csv", dtype={"settings_hash": str})
        vlog["origin"] = pd.to_datetime(vlog["origin"])
        vlog = vlog[vlog["origin"] <= pd.Timestamp(VALID_ORIGINS[-1])]           # validation only, nothing later
        vms = {m: vlog[vlog["model"] == m].reset_index(drop=True) for m in ("M1", "M2", "M3")}
        print("origins in the fitlog:", vlog["origin"].nunique(), "| settings hashes:", sorted(vlog["settings_hash"].unique()))
        tp = pd.read_csv(paths.CHECKPOINTS / "main_params.csv", dtype={"origin": str})
        tp["origin"] = pd.to_datetime(tp["origin"])
        tp = tp[tp["origin"] <= pd.Timestamp(VALID_ORIGINS[-1])]                 # validation only, nothing later


        def flag(series):
            """Column of True/False (read from csv, possibly as text) as booleans."""
            return series.astype(str).str.lower().eq("true")
else:
    print("skipped: this cell belongs to stage validation (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage validation (STAGE is 'freeze')


In [11]:
# ── CONVERGENCE, RESTARTS, RELABELING, DEGENERACY ─────────────────────────────────────────────
if on("validation"):
    if not (paths.CHECKPOINTS / "main_fitlog.csv").exists():
        print("skipped: " + str(paths.CHECKPOINTS / "main_fitlog.csv") + " does not exist yet (run its stage first)")
    else:
        rows = []
        for m, g in vms.items():
            rs = g["restart_reason"].fillna("")
            restarted = rs != ""
            rows.append({
                "model": m, "weeks": len(g),
                "converged": flag(g["converged"]).mean(),
                "refit_from_scratch": flag(g["refit_from_scratch"]).mean(),
                "relabeled": flag(g["relabeled"]).mean(),
                "degenerate (count)": int(flag(g["degenerate"]).sum()),
                "restart weeks": int(restarted.sum()),
                "share not 'scheduled'": float((restarted & (rs != "scheduled")).sum() / max(restarted.sum(), 1)),
                "share restarts that helped (LL gain > 1e-6)": float((g["ll_gain_from_restart"].fillna(0) > 1e-6).sum() / max(restarted.sum(), 1)),
            })
        display(pd.DataFrame(rows).set_index("model").round(3))
        print("restart reasons (all MS models):")
        display(pd.concat([g["restart_reason"].fillna("").replace("", "(no restart)") for g in vms.values()])
                .value_counts().rename("weeks").to_frame())
else:
    print("skipped: this cell belongs to stage validation (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage validation (STAGE is 'freeze')


In [12]:
# ── THE THREE SAFETY THRESHOLDS: WHERE DO THE NUMBERS SIT? ────────────────────────────────────
# SIGMA2_RATIO_MIN : sigma2_H / sigma2_N should be clearly above 1 (the turbulent regime must be more volatile)
# MIN_DURATION     : expected duration (weeks) of each regime should not be tiny
# VAR_FLOOR_FRAC   : smallest regime variance relative to the variance of returns (var(r)); we use the origin-by-origin
#                    expanding variance of the weekly return up to each origin.
if on("validation"):
    if not (paths.CHECKPOINTS / "main_fitlog.csv").exists():
        print("skipped: " + str(paths.CHECKPOINTS / "main_fitlog.csv") + " does not exist yet (run its stage first)")
    else:
        r = panel.set_index("date")["r"]
        varr = pd.Series({o: r[:o].var() for o in vlog["origin"].unique()})
        rows = []
        for m, g in vms.items():
            ratio = g["sigma2_H"] / g["sigma2_N"]
            floor = np.minimum(g["sigma2_N"], g["sigma2_H"]).values / varr.reindex(g["origin"]).values
            for lab, v in [("sigma2_H / sigma2_N", ratio), ("dur_N (weeks)", g["dur_N"]), ("dur_H (weeks)", g["dur_H"]),
                           ("min sigma2 / var(r)", pd.Series(floor))]:
                rows.append({"model": m, "quantity": lab, "min": v.min(), "5th pct": v.quantile(0.05), "median": v.median()})
        display(pd.DataFrame(rows).round(3).set_index(["model", "quantity"]))
        print(f"thresholds now in config.py: SIGMA2_RATIO_MIN = {C.SIGMA2_RATIO_MIN}, MIN_DURATION = {C.MIN_DURATION}, VAR_FLOOR_FRAC = {C.VAR_FLOOR_FRAC}")
        print("Look for: a 5th percentile or minimum that sits right next to a threshold (the threshold would be binding).")
else:
    print("skipped: this cell belongs to stage validation (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage validation (STAGE is 'freeze')


**What the next cell does (degenerate fits and slopes at the cap).** Sometimes the data let one combination of variables (VIX, GPR, realized volatility, Iran news) predict the calm-to-turbulent switch almost perfectly. The likelihood then keeps rising as those slopes grow without end (*near-separation*), and the estimates become meaningless numbers in the thousands. To stop that, every transition **slope** is capped at ±`MAX_TVTP_COEF` (= 50) during estimation. A slope that ends exactly at ±50 means "the data push this effect as far as allowed": its sign is informative, its size and standard error are not. We print, per model, the share of validation weeks that are *degenerate* (a regime too calm or too short-lived) and the share of weeks with **at least one slope at the ±50 cap**.

In [13]:
# ── DEGENERATE SHARE AND SLOPES AT THE CAP ────────────────────────────────────────────────────
# at_cap (fitlog) = number of transition slopes at +-MAX_TVTP_COEF in that week's fit (0 for M1: no slopes).
if on("validation"):
    if not (paths.CHECKPOINTS / "main_fitlog.csv").exists():
        print("skipped: " + str(paths.CHECKPOINTS / "main_fitlog.csv") + " does not exist yet (run its stage first)")
    else:
        rows = []
        for m, g in vms.items():
            n = len(g)
            cap = pd.to_numeric(g["at_cap"], errors="coerce").fillna(0)
            rows.append({"model": m, "weeks": n,
                         "degenerate share": flag(g["degenerate"]).sum() / n,
                         f"weeks with a slope at the cap (share)": float((cap > 0).sum()) / n,
                         "max slopes at the cap in one week": int(cap.max())})
        display(pd.DataFrame(rows).set_index("model").round(3))
        # which slopes hit the cap, and in how many validation weeks (from the params checkpoint, validation origins only)
        sl = tp[tp["name"].str.match(r"^p(00|10)_") & ~tp["name"].str.endswith("_const") & tp["model"].isin(["M2", "M3"])]
        hit = sl[sl["value"].abs() >= C.MAX_TVTP_COEF - 1e-6]
        print(f"slopes at the ±{C.MAX_TVTP_COEF:g} cap (validation weeks):",
              hit.groupby(["model", "name"]).size().to_dict() if len(hit) else "none")
else:
    print("skipped: this cell belongs to stage validation (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage validation (STAGE is 'freeze')


**What the next cell does (is the constrained M3 stable?).** In the `constrained` variant M3 is M2 plus one extra number (Iran news in the calm to turbulent switch). Setting that number to 0 gives back M2, so the best M3 fit can never have a lower likelihood than M2. If in a week M3 did not converge **or** its likelihood is below M2's, the optimizer got stuck, and we call that week "unstable". If more than 5% of weeks are unstable, the project rule says: switch to `iran_both_logits`.

In [14]:
# ── CONSTRAINED-M3 STABILITY AND THE VARIANT RULE ─────────────────────────────────────────────
if on("validation"):
    if not (paths.CHECKPOINTS / "main_fitlog.csv").exists():
        print("skipped: " + str(paths.CHECKPOINTS / "main_fitlog.csv") + " does not exist yet (run its stage first)")
    else:
        g3 = vms["M3"]
        ll2 = vms["M2"].set_index("origin")["loglike"]
        unstable = (~flag(g3["converged"])) | (g3["loglike"].values < ll2.reindex(g3["origin"]).values - 1e-6)
        share = float(unstable.mean())
        print(f"M3 variant now: {C.M3_VARIANT}")
        print(f"weeks where M3 is not converged or LL(M3) < LL(M2) - 1e-6: {int(unstable.sum())} of {len(g3)} = {share:.1%}")
        print("nesting_ok == False weeks (M3 vs M2):", int((~flag(g3['nesting_ok'])).sum()), "| (M2 vs M1):", int((~flag(vms['M2']['nesting_ok'])).sum()))
        if C.M3_VARIANT == "constrained" and share > 0.05:
            print("RECOMMENDATION: switch. Set  M3_VARIANT = \"iran_both_logits\"  in src/brent/config.py (by hand, after the team agrees), "
                  "then rerun the notebook from the setup cell (so the models are rebuilt) with this stage. "
                  "The settings hash changes, so validation reruns from the start (as long as the first validation run).")
        elif C.M3_VARIANT == "constrained":
            print("RECOMMENDATION: keep M3_VARIANT = \"constrained\" (unstable share is at most 5%).")
        else:
            print("Variant is already iran_both_logits: review the table again under this variant; no further switch is planned.")
else:
    print("skipped: this cell belongs to stage validation (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage validation (STAGE is 'freeze')


In [15]:
# ── BENCHMARK FITS: GARCH-X NON-CONVERGED, HAR FLOORED, GARCH-X LR ORDERING ───────────────────
if on("validation"):
    if not (paths.CHECKPOINTS / "main_fitlog.csv").exists():
        print("skipped: " + str(paths.CHECKPOINTS / "main_fitlog.csv") + " does not exist yet (run its stage first)")
    else:
        bl = vlog[~vlog["model"].isin(["M1", "M2", "M3"])]
        ran_b = bl[~bl["restart_reason"].fillna("").astype(str).str.startswith("skipped")]
        rows = []
        for m, g in ran_b.groupby("model", sort=False):
            rows.append({"model": m, "weeks run": len(g), "not converged": int((~flag(g["converged"])).sum()),
                         "har_floored weeks": int(flag(g["har_floored"]).sum()) if m.startswith("HAR") else ""})
        display(pd.DataFrame(rows).set_index("model"))

        # Likelihood ordering GARCH_t <= GARCHX_C <= GARCHX_CI. GARCH-X needs hourly data, so the first possible origin is
        # oos.first_origin("GARCHX_C") (not 2012-12-28, where no hourly data exists yet). Same-sample fits, so the order must hold.
        o1 = oos.first_origin("GARCHX_C")
        g_c = I.garchx_lr(panel, o1, None, tuple(C.X_CONTROLS))
        g_ci = I.garchx_lr(panel, o1, tuple(C.X_CONTROLS), tuple(C.X_CONTROLS) + (C.IRAN_VAR,))
        print(f"\nGARCH-X likelihoods at origin {o1.date()}: GARCH_t {g_c['ll_small']:.3f} <= GARCHX_C {g_c['ll_big']:.3f} <= GARCHX_CI {g_ci['ll_big']:.3f}")
        print("ordering holds:", g_c["ll_small"] <= g_c["ll_big"] + 1e-6 <= g_ci["ll_big"] + 2e-6)
else:
    print("skipped: this cell belongs to stage validation (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage validation (STAGE is 'freeze')


### What this means (practice round, 2013–2018, 313 weeks)
- **Every fit converged**, for every model in every week.
- **First run → a problem, then a fix.** The first validation run found overfitting in 2017–2018: in 95 (M2) and 84 (M3) weeks the calm → turbulent coefficients blew up to values as large as 10,382, turning the model into an on/off switch. The team decided to **cap these coefficients at ±50**. After rerunning: **0 degenerate fits**; in 96 (M2) and 87 (M3) weeks, all in 2017–2018, a coefficient sits at the cap. This is logged and reported as a limitation.
- **M3 is stable:** 0% of weeks where M3 failed to converge or fitted worse than M2. The project rule (switch if more than 5%) says: **keep the `constrained` M3** (Iran news only affects calm → turbulent).
- **Safety limits are far from being hit:** the turbulent regime always has at least 4× the variance of the calm one (the limit is 1.5×), and the shortest turbulent spell is 3.4 weeks (the limit is 2).
- **Restarts are worth it:** 77 restarts, 63 of which found a better fit.
- **Hourly benchmarks** converged in every week from their start dates; the HAR safety floor was needed in 1 week.
- **Worth knowing:** in M3 the calm regime almost never ends *at average conditions*. Switches into turbulence happen only when GPR, VIX or volatility are unusually high: the model relies entirely on its risk measures.

### Checkpoint 2: agree the settings as a team

The review above is the evidence. The values to agree are: `SEARCH_REPS_FIRST, N_RESTARTS, RESTART_EVERY, SIGMA2_RATIO_MIN, VAR_FLOOR_FRAC, MIN_DURATION, M3_VARIANT` (they are all in `config.py` now). **Consequence:** if any of them changes after this validation run, the settings hash changes and the `full` stage throws the validation checkpoint away and reruns from Dec 28 2012 (about 40% more runtime).

## 3 · Stage `freeze`: write the agreed settings down

**What the next cells do.**
1. Check that you set `SPEC_FROZEN = True` in `config.py` (by hand). This notebook never edits `config.py`.
2. Write `results/frozen/spec_freeze.json` with every frozen setting, the settings hash and the git commit.
3. Say whether the `full` stage will **resume** from the validation run (hash unchanged) or **rerun** from the start.

In [16]:
# ── WRITE spec_freeze.json ────────────────────────────────────────────────────────────────────
if STAGE == "freeze":
    if not C.SPEC_FROZEN:
        print("NOT FROZEN YET. Agree the values as a team, then set this line by hand in src/brent/config.py:")
        print("    SPEC_FROZEN = True")
        print("and run this stage again.")
    else:
        h = oos.settings_hash(MS, C.BENCHMARKS_MAIN, C.IRAN_VAR)
        freeze = {"phi_switching": C.PHI_SWITCHING, "SEARCH_REPS_FIRST": C.SEARCH_REPS_FIRST, "N_RESTARTS": C.N_RESTARTS,
                  "RESTART_EVERY": C.RESTART_EVERY, "SIGMA2_RATIO_MIN": C.SIGMA2_RATIO_MIN,
                  "VAR_FLOOR_FRAC": C.VAR_FLOOR_FRAC, "MIN_DURATION": C.MIN_DURATION,
                  "MAX_TVTP_COEF": C.MAX_TVTP_COEF, "CONTROLS_ONLY_NH": C.CONTROLS_ONLY_NH,
                  "M3_VARIANT": C.M3_VARIANT, "INTRADAY_ROLL_RULE": C.INTRADAY_ROLL_RULE,
                  "INTRADAY_REDISTRIBUTE": C.INTRADAY_REDISTRIBUTE, "HOURLY_SAMPLE_START": C.HOURLY_SAMPLE_START,
                  "MAIN_RV": C.MAIN_RV, "X_RV_COL": C.X_RV_COL, "X_SAMPLE_START": C.X_SAMPLE_START,
                  "BENCHMARKS_MAIN": list(C.BENCHMARKS_MAIN), "settings_hash": h, "git_hash": paths.git_hash()}
        paths.FROZEN.mkdir(parents=True, exist_ok=True)
        (paths.FROZEN / "spec_freeze.json").write_text(json.dumps(freeze, indent=2, default=str))
        print(json.dumps(freeze, indent=2, default=str))
        f = paths.CHECKPOINTS / "main_fitlog.csv"
        if f.exists():
            ck = pd.read_csv(f, dtype={"settings_hash": str, "origin": str})
            hashes = set(ck["settings_hash"])
            n_valid = ck.loc[ck["origin"] <= VALID_ORIGINS[-1].strftime("%Y-%m-%d"), "origin"].nunique()
            if hashes != {h}:
                print(f"\nfull will RERUN from 2012-12-28 (fitlog hashes {sorted(hashes)} differ from {h})")
            elif n_valid < len(VALID_ORIGINS):
                print(f"\nthe validation checkpoint has only {n_valid} of {len(VALID_ORIGINS)} origins: "
                      "full will RESUME from where it stopped (the remaining validation weeks run first)")
            else:
                print(f"\nfull will RESUME from the validation run (all {n_valid} validation origins are saved)")
        else:
            print("\nno validation checkpoint on disk: full will start from 2012-12-28")
else:
    print("skipped: this cell belongs to stage freeze (STAGE is " + repr(STAGE) + ")")

{
  "phi_switching": false,
  "SEARCH_REPS_FIRST": 30,
  "N_RESTARTS": 3,
  "RESTART_EVERY": 13,
  "SIGMA2_RATIO_MIN": 1.5,
  "VAR_FLOOR_FRAC": 0.05,
  "MIN_DURATION": 2.0,
  "MAX_TVTP_COEF": 50,
  "CONTROLS_ONLY_NH": true,
  "M3_VARIANT": "constrained",
  "INTRADAY_ROLL_RULE": {
    "kind": "gap",
    "max_gap_hours": 84
  },
  "INTRADAY_REDISTRIBUTE": false,
  "HOURLY_SAMPLE_START": "2013-06-07",
  "MAIN_RV": "rv_hourly",
  "X_RV_COL": "rv_hourly",
  "X_SAMPLE_START": "2013-06-07",
  "BENCHMARKS_MAIN": [
    "GARCH_t",
    "HAR",
    "RW",
    "HAR_h",
    "HARX_C",
    "HARX_CI",
    "GARCHX_C",
    "GARCHX_CI"
  ],
  "settings_hash": "d529415f6ba85431",
  "git_hash": "b6ef137"
}

full will RESUME from the validation run (all 313 validation origins are saved)


### What this means
- **The settings are locked**, unchanged after the practice round, in `results/frozen/spec_freeze.json` (fingerprint `d529415f6ba85431`), and `SPEC_FROZEN = True` in `config.py`.
- From here on, the 2019–2026 forecasts can only be made with exactly these settings. If any setting changed, the fingerprint would change and the code would refuse to continue, or start over from scratch.
- Because nothing changed, the full run could **continue** from the 313 finished practice weeks.

## 4 · Stage `full`: all 723 origins

**What the next cells do.**
1. **Guard:** the spec must be frozen and the settings hash must equal the one in `spec_freeze.json`; otherwise we stop.
2. If `forecasts_main.csv` already exists (and `FORCE_OOS = False`) we **skip** the run and just load the saved files. Otherwise we run `oos.run_oos(panel, "main", MS)` over all origins (2012-12-28 to 2026-10-30; it resumes from the validation run if the hash matches) and then `oos.assemble("main")` writes the three output files. The validation, test and holdout weeks come out of the same pass; the split is assigned by target date.

To start the long run from a terminal (so the notebook stays free):

```
BRENT_STAGE=full uv run jupyter nbconvert --to notebook --execute --inplace notebooks/05_oos_forecasts.ipynb
```

If it is interrupted, run the same command again: it resumes.

In [17]:
# ── GUARD: SPEC FROZEN AND HASH MATCHES ───────────────────────────────────────────────────────
if on("full"):
    if not (paths.FROZEN / "spec_freeze.json").exists():
        print("skipped: " + str(paths.FROZEN / "spec_freeze.json") + " does not exist yet (run its stage first)")
    else:
        fz = json.loads((paths.FROZEN / "spec_freeze.json").read_text())
        H_NOW = oos.settings_hash(MS, C.BENCHMARKS_MAIN, C.IRAN_VAR)
        assert C.SPEC_FROZEN, "SPEC_FROZEN is False in config.py: freeze the specification first (stage 'freeze')"
        assert fz["settings_hash"] == H_NOW, f"settings changed since the freeze: {fz['settings_hash']} vs {H_NOW}"
        print("OK: spec is frozen, hash", H_NOW)
else:
    print("skipped: this cell belongs to stage full (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage full (STAGE is 'freeze')


In [18]:
# ── RUN OR LOAD ───────────────────────────────────────────────────────────────────────────────
if on("full"):
    FC_FILE = paths.FORECASTS / "forecasts_main.csv"
    if FC_FILE.exists() and not FORCE_OOS:
        print(f"skip: {FC_FILE.name} already exists (set FORCE_OOS = True to rebuild it)")
    elif STAGE == "all":
        print("skipped: STAGE is 'all' (display only) and forecasts_main.csv does not exist yet; run stage 'full' first")
    else:
        t0 = time.perf_counter()
        oos.run_oos(panel, "main", MS)                       # all 723 origins; resumes if the hash matches
        oos.assemble("main")                                 # writes forecasts_main.csv, params_main.csv, fitlog_main.csv
        print(f"full run + assembly: {(time.perf_counter() - t0) / 60:.1f} min")
    if FC_FILE.exists():                                     # load the saved outputs (the checks below use them)
        fc = pd.read_csv(FC_FILE, parse_dates=["date", "origin"])
        fitlog = pd.read_csv(paths.FITS / "fitlog_main.csv", parse_dates=["origin"])
        params = pd.read_csv(paths.FITS / "params_main.csv", parse_dates=["origin"])
        print(f"forecasts: {len(fc):,} rows | fitlog: {len(fitlog):,} rows | params: {len(params):,} rows")
else:
    print("skipped: this cell belongs to stage full (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage full (STAGE is 'freeze')


### What this means
- **Every forecast was made:** 723 weekly forecasts for each model (the hourly-data models from their start dates: 636 for HAR_h and the HAR-X models, 596 for the GARCH-X models). That is 13,945 rows in `results/forecasts/forecasts_main.csv`.
- **The three periods:** 313 practice weeks (2013–2018), 365 test weeks (2019–2025) and 45 holdout weeks (2026). Test and holdout were forecast in one pass with the locked settings.
- **Time:** 6.4 minutes for the whole run.
- **Next:** notebook 06 scores these forecasts against what actually happened, to see whether the models with Iran news forecast better.

## 5 · Checks

**What the next cell does.** Every line is an `assert`. If one fails it prints the offending rows. The checks: the right number of forecast weeks per model and target (see the table in section 0), the split counts 313 / 365 / 45, the origin is exactly 7 days before the target, no forecast is for a week at or before its origin, variances are positive, probabilities are between 0 and 1, and every row carries the frozen settings hash. Then it prints the convergence summary and the nesting counts.

In [19]:
# ── CHECKS ────────────────────────────────────────────────────────────────────────────────────
if on("full"):
    if not (paths.FORECASTS / "forecasts_main.csv").exists():
        print("skipped: " + str(paths.FORECASTS / "forecasts_main.csv") + " does not exist yet (run its stage first)")
    else:
        def check(cond, msg, rows=None):
            """assert with the offending rows printed first."""
            if not cond:
                if rows is not None:
                    display(rows.head(10))
                raise AssertionError(msg)


        # 1. number of forecast weeks per model and target (counts from oos.first_origin; MS and RW/GARCH_t/HAR have all 723)
        TARGETS = {"M1": ["var", "r", "p_high", "p_exceed"], "M2": ["var", "r", "p_high", "p_exceed"], "M3": ["var", "r", "p_high", "p_exceed"],
                   "RW": ["r"], **{m: ["var"] for m in C.BENCHMARKS_MAIN if m != "RW"}}
        cnt = fc.groupby(["model", "target"])["date"].nunique()
        for m, tg in TARGETS.items():
            for t in tg:
                check(cnt.get((m, t), 0) == EXPECTED[m], f"{m}/{t}: {cnt.get((m, t), 0)} weeks, expected {EXPECTED[m]}")
        check(set(fc["model"]) == set(TARGETS), f"unexpected models: {set(fc['model']) ^ set(TARGETS)}")
        print("OK: forecast weeks per model/target match", {m: EXPECTED[m] for m in TARGETS})

        # 2. split counts (by target date, taken from the MS model M1 which has all weeks)
        sp = fc[(fc["model"] == "M1") & (fc["target"] == "var")]["split"].value_counts().to_dict()
        check(sp == {"validation": 313, "test": 365, "holdout": 45}, f"split counts {sp}")
        print("OK: split counts", sp)

        # 3. timing of origin and target
        gap = (fc["date"] - fc["origin"]).dt.days
        check((gap == 7).all(), "origin is not date - 7 days", fc[gap != 7])
        check((fc["date"] > fc["origin"]).all(), "a target is not after its origin", fc[fc["date"] <= fc["origin"]])
        print("OK: origin = target - 7 days for every row")

        # 4. values
        v = fc[fc["target"] == "var"]
        check((v["forecast"] > 0).all(), "non-positive variance forecast", v[v["forecast"] <= 0])
        ph = fc[fc["target"] == "p_high"]
        check(ph["forecast"].between(0, 1).all(), "p_high outside [0, 1]", ph[~ph["forecast"].between(0, 1)])
        pe = fc[fc["target"] == "p_exceed"]
        check(pe["forecast"].between(0, 1).all(), "p_exceed outside [0, 1]", pe[~pe["forecast"].between(0, 1)])
        check(np.isfinite(fc["forecast"]).all(), "NaN or infinite forecast", fc[~np.isfinite(fc["forecast"])])
        print("OK: all var > 0, p_high and p_exceed in [0, 1], no NaN")

        # 5. every row carries the frozen hash (assembled files drop the column, so check the checkpoints)
        for k in ("forecasts", "params", "fitlog"):
            ck = pd.read_csv(paths.CHECKPOINTS / f"main_{k}.csv", dtype=str) if (paths.CHECKPOINTS / f"main_{k}.csv").exists() else None
            if ck is None:
                print(f"note: no {k} checkpoint on disk (committed outputs only): hash check skipped for it")
                continue
            check((ck["settings_hash"] == fz["settings_hash"]).all(), f"{k} checkpoint has rows with another settings hash",
                  ck[ck["settings_hash"] != fz["settings_hash"]])
        print("OK: checkpoint rows carry the frozen hash", fz["settings_hash"])
else:
    print("skipped: this cell belongs to stage full (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage full (STAGE is 'freeze')


In [20]:
# ── CONVERGENCE SUMMARY AND NESTING COUNTS ────────────────────────────────────────────────────
if on("full"):
    if not (paths.FORECASTS / "forecasts_main.csv").exists():
        print("skipped: " + str(paths.FORECASTS / "forecasts_main.csv") + " does not exist yet (run its stage first)")
    else:
        ran_f = fitlog[~fitlog["restart_reason"].fillna("").astype(str).str.startswith("skipped")].copy()
        for c in ("converged", "refit_from_scratch", "degenerate", "har_floored"):
            ran_f[c] = ran_f[c].astype(str).str.lower().eq("true")
        summ = ran_f.groupby("model", sort=False).agg(weeks=("origin", "size"), convergence_rate=("converged", "mean"),
                                                      not_converged=("converged", lambda s: int((~s).sum())),
                                                      refit_from_scratch=("refit_from_scratch", "sum"),
                                                      degenerate=("degenerate", "sum"), har_floored=("har_floored", "sum"))
        display(summ.round(3))
        failed = fitlog[fitlog["restart_reason"].fillna("").astype(str).str.startswith("fit_failed")]
        print("fits that raised an error (last week carried forward / previous forecast reused):",
              failed.groupby("model").size().to_dict() if len(failed) else "none")

        # nesting per week (MS ladder): LL(M3) < LL(M2) - 1e-6 and LL(M2) < LL(M1) - 1e-6
        ll = fitlog[fitlog["model"].isin(["M1", "M2", "M3"])].pivot(index="origin", columns="model", values="loglike")
        print("weeks with LL(M3) < LL(M2) - 1e-6:", int((ll["M3"] < ll["M2"] - 1e-6).sum()), "of", len(ll))
        print("weeks with LL(M2) < LL(M1) - 1e-6:", int((ll["M2"] < ll["M1"] - 1e-6).sum()), "of", len(ll))
        llb = fitlog[fitlog["model"].isin(["GARCHX_C", "GARCHX_CI"])].pivot(index="origin", columns="model", values="loglike").dropna()
        print("weeks with LL(GARCHX_CI) < LL(GARCHX_C) - 1e-6 (warm-started, so printed, not asserted):",
              int((llb["GARCHX_CI"] < llb["GARCHX_C"] - 1e-6).sum()), "of", len(llb))

        # at the first GARCH-X origin the ordering GARCH_t <= GARCHX_C <= GARCHX_CI must hold (same sample, same fit routine)
        o1 = oos.first_origin("GARCHX_C")
        g_c = I.garchx_lr(panel, o1, None, tuple(C.X_CONTROLS))
        g_ci = I.garchx_lr(panel, o1, tuple(C.X_CONTROLS), tuple(C.X_CONTROLS) + (C.IRAN_VAR,))
        assert g_ci["ll_big"] >= g_c["ll_big"] - 1e-6 >= g_c["ll_small"] - 2e-6, "GARCH-X likelihood ordering fails at the first origin"
        print(f"OK: at {o1.date()}: GARCH_t {g_c['ll_small']:.3f} <= GARCHX_C {g_c['ll_big']:.3f} <= GARCHX_CI {g_ci['ll_big']:.3f}")
else:
    print("skipped: this cell belongs to stage full (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage full (STAGE is 'freeze')


**What the next cell does (resume check).** Proves that stopping and restarting gives the *same* numbers as one uninterrupted run. We run the first 8 validation origins in one go (`resume_ref`), and the same 8 in two pieces (`resume_chk`: 4 origins, then the rest). We assemble both, assert that forecasts and parameters are equal (ignoring the run name and the timing), and delete the test files. It takes about a minute, so it only runs under `BRENT_STAGE=full` (that is how the final version of this notebook is run).

In [21]:
# ── RESUME IDENTITY ───────────────────────────────────────────────────────────────────────────
# Only for STAGE == "full" (a few minutes of fitting); under "all" it is skipped.
if on("full"):
    if not (paths.FORECASTS / "forecasts_main.csv").exists():
        print("skipped: " + str(paths.FORECASTS / "forecasts_main.csv") + " does not exist yet (run its stage first)")
    else:
        if STAGE == "full":
            o8 = VALID_ORIGINS[:8]
            oos.run_oos(panel, "resume_ref", MS, origins=o8, force=True)                 # one go
            oos.run_oos(panel, "resume_chk", MS, origins=o8, force=True, max_origins=4)  # interrupted after 4 origins
            oos.run_oos(panel, "resume_chk", MS, origins=o8)                             # resumed to the end
            a_fc, a_par, a_log = oos.assemble("resume_ref")
            b_fc, b_par, b_log = oos.assemble("resume_chk")
            pd.testing.assert_frame_equal(a_fc.drop(columns="run"), b_fc.drop(columns="run"), check_exact=True)
            pd.testing.assert_frame_equal(a_par, b_par, check_exact=True)
            print("OK: an interrupted-and-resumed run gives identical forecasts and parameters")
            for pat in ("forecasts_resume_*", "params_resume_*", "fitlog_resume_*"):         # clean up the test files
                for d in (paths.FORECASTS, paths.FITS):
                    for f in d.glob(pat):
                        f.unlink()
            for f in paths.CHECKPOINTS.glob("resume_*"):
                f.unlink()
        else:
            print("skipped: resume check runs only under BRENT_STAGE=full")
else:
    print("skipped: this cell belongs to stage full (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage full (STAGE is 'freeze')


In [22]:
# ── ACTUALS AND SIZE BUDGET ───────────────────────────────────────────────────────────────────
if on("full"):
    if not (paths.FORECASTS / "forecasts_main.csv").exists():
        print("skipped: " + str(paths.FORECASTS / "forecasts_main.csv") + " does not exist yet (run its stage first)")
    else:
        lf = oos.load_forecasts("main")
        sub = lf[(lf["date"] >= pd.Timestamp(C.HOURLY_SAMPLE_START)) & (lf["date"] <= "2026-12-31")]
        assert sub["actual_rv_hourly"].notna().all(), "NaN actual_rv_hourly after the hourly sample starts"
        print("OK: no NaN actual_rv_hourly from", sub["date"].min().date(), "on")
        paths.check_budget()
else:
    print("skipped: this cell belongs to stage full (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage full (STAGE is 'freeze')


### What this means
- **Every check passed:**
  - the right number of forecasts for every model;
  - 313 / 365 / 45 weeks in practice / test / holdout;
  - every forecast for a week was made from data up to the Friday before (origin = target − 7 days);
  - all variances positive, all probabilities between 0 and 1, no gaps;
  - every row carries the locked fingerprint.
- **Every fit converged; no fit failed or collapsed.** Each bigger model fitted at least as well as the smaller one in **every** week (0 violations of LL(M3) ≥ LL(M2) ≥ LL(M1)).
- GARCH-X *with* Iran fitted slightly worse than without it in 15 of 596 weeks. That happens because each week starts from last week's answer. It is printed, not an error, and the check in its first week is in the right order.
- **The HAR safety floor** (it stops negative variance forecasts) was needed in only a few weeks: 10 (HAR), 1 (HAR_h) and 4 (HARX_CI).
- **Resume test:** stopping and restarting the run gives exactly the same numbers.
- **Cap:** M2 and M3 have a coefficient at the ±50 cap in 63 test weeks each (and 96 / 87 practice weeks). We report this as a limitation.

## 6 · Runtime and convergence figure

**What the next cell does.** Top panel: seconds per fit for each Markov-switching model, by origin, with the restart weeks marked as dots. Bottom panel: a 26-week rolling share of weeks where the fit did **not** converge or was degenerate. This shows whether the fits get harder at some period.

In [23]:
# ── FIGURE: RUNTIME AND CONVERGENCE ───────────────────────────────────────────────────────────
if on("full"):
    if not (paths.FORECASTS / "forecasts_main.csv").exists():
        print("skipped: " + str(paths.FORECASTS / "forecasts_main.csv") + " does not exist yet (run its stage first)")
    else:
        fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
        for m in ("M1", "M2", "M3"):
            g = fitlog[fitlog["model"] == m].sort_values("origin")
            line, = ax[0].plot(g["origin"], g["seconds"], lw=0.8, label=m)
            rest = g[g["restart_reason"].fillna("") != ""]
            ax[0].scatter(rest["origin"], rest["seconds"], s=8, color=line.get_color(), zorder=3)
            bad = (~g["converged"].astype(str).str.lower().eq("true")) | g["degenerate"].astype(str).str.lower().eq("true")
            ax[1].plot(g["origin"], bad.astype(float).rolling(26, min_periods=1).mean(), label=m)
        ax[0].set_ylabel("seconds per fit"); ax[0].set_title("Fit time by origin (dots = weeks with restarts)"); ax[0].legend()
        ax[1].set_ylabel("share, rolling 26 weeks"); ax[1].set_title("Not converged or degenerate"); ax[1].set_xlabel("origin (Friday)")
        fig.tight_layout()
        plots.save_fig(fig, "oos_runtime_and_convergence")
        plt.show()
else:
    print("skipped: this cell belongs to stage full (STAGE is " + repr(STAGE) + ")")

skipped: this cell belongs to stage full (STAGE is 'freeze')


### What this means
- **Top panel:** a normal weekly fit takes 0.05–0.2 seconds. The dots are restart weeks (every 13 weeks) at 0.5–2 seconds; the very first fit (Dec 2012) is the slowest at about 2.5 seconds.
- **Bottom panel:** the share of failed or collapsed fits is **0 for the whole of 2013–2026**.
- **So:** the computing side is clean. Any differences between the models in notebook 06 come from the models themselves, not from fitting problems.

## Last cell: file-size budget

The checks cell above already ends with `paths.check_budget()`. Nothing in this notebook writes large files; the checkpoints in `results/checkpoints/` are git-ignored.

In [24]:
# ── BUDGET ────────────────────────────────────────────────────────────────────────────────────
paths.check_budget()

,rule,path,bytes,limit
